In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("SCRAPINGDOG_API_KEY")

In [7]:

import time
import random
from typing import Dict, Any, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import requests
import pandas as pd
from tqdm.auto import tqdm

# -----------------------------
# Config
# -----------------------------

if not API_KEY:
    raise RuntimeError("Missing SCRAPINGDOG_API_KEY env var. Set it before running.")

BASE_URL = "https://api.scrapingdog.com/amazon/search"
DEFAULT_PARAMS = {
    "api_key": API_KEY,
    "country": "us",
    "domain": "com",
    "premium": "true",
}

TARGET_UNIQUE_ASINS = 100_000

MAX_PAGES_PER_QUERY = 25
EMPTY_PAGE_STOP = 1  # stop after N consecutive page-windows with 0 NEW ASINs

# Concurrency
MAX_WORKERS = 5
PAGE_WINDOW = 5  # number of pages to fetch concurrently (should match workers for simplicity)

# Retry/backoff
MAX_RETRIES = 6
BACKOFF_BASE = 1.5

# Optional pacing (still useful even w/ threads)
BASE_SLEEP_SECONDS = 0.05
JITTER_SECONDS = 0.10


# -----------------------------
# Keywords: 3 big categories
# -----------------------------
KEYWORDS = {
    "computers": [
        "mechanical keyboard", "wireless keyboard", "wireless mouse", "gaming mouse", "laptop stand",
        "external hard drive", "ssd drive", "usb hub", "usb c adapter", "monitor stand", "computer monitor",
        "gaming monitor", "laptop charger", "webcam", "wireless headset", "microphone usb", "docking station",
        "graphics card", "cpu cooler", "gaming laptop", "chromebook", "desktop pc", "pc case", "power supply unit",
        "motherboard", "ram memory", "mechanical gaming keyboard",
    ],
    "mobile_phones": [
        "smartphone", "android phone", "iphone case", "wireless charger", "portable charger", "screen protector",
        "phone holder car", "selfie stick", "usb c cable", "lightning cable", "bluetooth headset", "phone tripod",
        "phone camera lens", "power bank", "magsafe charger", "wireless earbuds",
    ],
    "audio_headphones": [
        "bluetooth headphones", "noise cancelling headphones", "wireless earbuds", "wired earbuds",
        "over ear headphones", "studio monitor headphones", "gaming headset", "portable speaker", "soundbar",
        "home theater system", "subwoofer", "car speakers", "karaoke machine", "digital piano headphones",
    ],
    "tv": [
        "smart tv", "4k tv", "oled tv", "roku streaming stick", "fire tv stick", "apple tv", "home theater projector",
        "projector screen", "av receiver", "universal remote", "tv wall mount", "surround sound system",
        "blu ray player", "dvd player"
    ],
    "gaming": [
        "playstation 5", "xbox series x", "nintendo switch", "gaming chair", "gaming desk", "vr headset",
        "gaming controller", "racing wheel", "gaming mouse pad", "gaming microphone", "capture card",
    ],
    "photography": [
        "digital camera", "dslr camera", "mirrorless camera", "instant camera", "action camera", "camcorder",
        "camera tripod", "camera lens", "memory card", "camera bag", "ring light", "gimbal stabilizer",
        "camera microphone",
    ],
    "smart_devices": [
        "apple watch", "samsung galaxy watch", "fitness tracker", "smart band", "smartwatch charger",
        "vr glasses", "ar glasses", "smart ring",
    ],
    "smart_home": [
        "smart speaker", "echo dot", "google nest", "smart thermostat", "wifi plug", "smart light bulb",
        "video doorbell", "security camera", "baby monitor", "home security system", "wifi extender",
        "mesh wifi", "smart lock",
    ],
    "other": [
        "calculator", "e reader", "kindle", "drone", "robot vacuum", "dash cam", "car gps", "car stereo",
        "portable dvd player", "cb radio", "walkie talkie", "electronic dictionary", "electric shaver",
        "hair clipper", "massage gun"
    ],
}





QUERY_SCHEDULE = [(cat, q) for cat, qs in KEYWORDS.items() for q in qs]
print("Total V2 queries:", len(QUERY_SCHEDULE))


# -----------------------------
# Parsing helpers
# -----------------------------
def _parse_colors(colors: Any) -> Tuple[Optional[str], int]:
    if not isinstance(colors, list) or not colors:
        return None, 0
    titles = []
    for c in colors:
        if isinstance(c, dict):
            t = c.get("title")
            if t:
                titles.append(str(t))
    return (", ".join(titles), len(titles)) if titles else (None, 0)


def flatten_product(
    product: Dict[str, Any],
    *,
    category: str,
    query: str,
    page: int,
    source_section: str,
    location: Optional[str],
    search_message: Optional[str],
) -> Dict[str, Any]:
    colors_str, num_colors = _parse_colors(product.get("colors"))
    return {
        "asin": product.get("asin"),
        "category": category,
        "query": query,
        "page": page,
        "source_section": source_section,
        "type": product.get("type"),
        "title": product.get("title"),
        "image": product.get("image"),
        "has_prime": product.get("has_prime"),
        "is_best_seller": product.get("is_best_seller"),
        "is_amazon_choice": product.get("is_amazon_choice"),
        "limited_time_deal": product.get("limited_time_deal"),
        "deal_of_the_day": product.get("deal_of_the_day"),
        "stars": product.get("stars"),
        "total_reviews": product.get("total_reviews"),
        "url": product.get("url"),
        "optimized_url": product.get("optimized_url"),
        "sponsored": product.get("sponsored"),
        "number_of_people_bought": product.get("number_of_people_bought"),
        "delivery": product.get("delivery"),
        "availability_quantity": product.get("availability_quantity"),
        "price_string": product.get("price_string"),
        "price_symbol": product.get("price_symbol"),
        "price": product.get("price"),
        "absolute_position": product.get("absolute_position"),
        "organic_position": product.get("organic_position"),
        "certification": product.get("certification"),
        "coupon_text": product.get("coupon_text"),
        "colors": colors_str,
        "num_colors": num_colors,
        "location": location,
        "search_message": search_message,
        "fetched_at_unix": int(time.time()),
    }


# -----------------------------
# Robust fetch (thread-safe)
# -----------------------------
def fetch_search_page(session: requests.Session, query: str, page: int) -> Dict[str, Any]:
    params = dict(DEFAULT_PARAMS)
    params.update({"query": query, "page": page})

    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(BASE_URL, params=params, timeout=60)
            if resp.status_code == 200:
                return resp.json()

            if resp.status_code in (429, 500, 502, 503, 504):
                sleep_s = (BACKOFF_BASE ** attempt) + random.random()
                time.sleep(sleep_s)
                last_err = RuntimeError(f"HTTP {resp.status_code}: {resp.text[:200]}")
                continue

            raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:500]}")

        except (requests.Timeout, requests.ConnectionError) as e:
            last_err = e
            sleep_s = (BACKOFF_BASE ** attempt) + random.random()
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed after {MAX_RETRIES} retries. Last error: {last_err}")


def fetch_and_parse(
    query: str,
    category: str,
    page: int,
    include_sponsored_brand_videos: bool,
) -> Tuple[int, List[Dict[str, Any]]]:
    """
    Returns: (page, rows)
    Create a fresh Session per task to avoid any thread-safety surprises.
    """
    # small jitter to smooth bursts
    time.sleep(BASE_SLEEP_SECONDS + random.random() * JITTER_SECONDS)

    with requests.Session() as session:
        data = fetch_search_page(session, query, page)

    location = data.get("location")
    search_message = data.get("search_message")

    rows: List[Dict[str, Any]] = []

    # main results
    for product in data.get("results", []) or []:
        if product.get("type") != "search_product":
            continue
        asin = product.get("asin")
        if not asin:
            continue
        rows.append(
            flatten_product(
                product,
                category=category,
                query=query,
                page=page,
                source_section="results",
                location=location,
                search_message=search_message,
            )
        )

    # sponsored videos (optional)
    if include_sponsored_brand_videos:
        for product in data.get("sponsored_brand_videos", []) or []:
            asin = product.get("asin")
            if not asin:
                continue
            rows.append(
                flatten_product(
                    product,
                    category=category,
                    query=query,
                    page=page,
                    source_section="sponsored_brand_videos",
                    location=location,
                    search_message=search_message,
                )
            )

    return page, rows


# -----------------------------
# Main multithreaded collector
# -----------------------------
def collect_products_multithreaded(
    target_unique_asins: int = TARGET_UNIQUE_ASINS,
    include_sponsored_brand_videos: bool = True,
    checkpoint_path: str = "asin_products_checkpoint.parquet",
) -> pd.DataFrame:
    # Load checkpoint
    if os.path.exists(checkpoint_path):
        df = pd.read_parquet(checkpoint_path)
        seen_asins = set(df["asin"].dropna().astype(str).tolist())
    else:
        df = pd.DataFrame()
        seen_asins = set()

    lock = Lock()

    outer = tqdm(total=target_unique_asins, initial=len(seen_asins), desc="Unique ASINs collected")

    def commit_rows(rows: List[Dict[str, Any]]) -> int:
        """Thread-safe dedupe + append. Returns number of NEW ASINs added."""
        nonlocal df
        new_rows = []
        with lock:
            for r in rows:
                asin = r.get("asin")
                if not asin:
                    continue
                asin = str(asin)
                if asin in seen_asins:
                    continue
                seen_asins.add(asin)
                new_rows.append(r)

            if new_rows:
                df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

                # checkpoint every ~500 new unique ASINs
                if (len(seen_asins) % 500) < len(new_rows):
                    df.to_parquet(checkpoint_path, index=False)

        return len(new_rows)

    try:
        for category, q in QUERY_SCHEDULE:
            if len(seen_asins) >= target_unique_asins:
                break

            empty_streak = 0
            pbar_q = tqdm(total=MAX_PAGES_PER_QUERY, desc=f"{category} | {q}", leave=False)

            page = 1
            while page <= MAX_PAGES_PER_QUERY and len(seen_asins) < target_unique_asins:
                # create a window of pages to fetch concurrently
                pages = list(range(page, min(page + PAGE_WINDOW, MAX_PAGES_PER_QUERY + 1)))

                window_new_asins = 0
                window_rows_count = 0

                with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
                    futures = [
                        ex.submit(fetch_and_parse, q, category, p, include_sponsored_brand_videos)
                        for p in pages
                    ]

                    for fut in as_completed(futures):
                        try:
                            p, rows = fut.result()
                        except Exception as e:
                            # If a page fails, treat it as empty for stop logic,
                            # but continue the rest of the window.
                            rows = []
                        window_rows_count += len(rows)
                        added = commit_rows(rows)
                        window_new_asins += added
                        if added:
                            outer.update(added)

                # progress UI
                pbar_q.update(len(pages))
                pbar_q.set_postfix({
                    "page_end": pages[-1],
                    "rows": window_rows_count,
                    "new_asins": window_new_asins,
                    "unique_total": len(seen_asins),
                    "empty_streak": empty_streak,
                })

                # early stop per query if we stop finding NEW ASINs
                if window_new_asins == 0:
                    empty_streak += 1
                else:
                    empty_streak = 0

                if empty_streak >= EMPTY_PAGE_STOP:
                    break

                page += PAGE_WINDOW

            pbar_q.close()

        df.to_parquet(checkpoint_path, index=False)
        return df

    finally:
        outer.close()


if __name__ == "__main__":
    df = collect_products_multithreaded(
        target_unique_asins=20000,
        include_sponsored_brand_videos=True,
        checkpoint_path="asin_products_checkpoint_6.parquet",
    )
    print("Done.")
    print("Rows:", len(df), "Unique ASINs:", df["asin"].nunique())
    df.to_csv("asin_products_old_words.csv", index=False)


Total V2 queries: 131


Unique ASINs collected:   0%|          | 88/20000 [00:06<23:49, 13.93it/s]

Unique ASINs collected:   1%|          | 167/20000 [00:09<11:13, 29.44it/s]

Unique ASINs collected:   1%|          | 216/20000 [00:13<14:27, 22.80it/s]

Unique ASINs collected:   1%|▏         | 275/20000 [00:16<11:18, 29.06it/s]

C:\Users\dsuni\AppData\Local\Temp\ipykernel_15092\3407218874.py:293: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
Unique ASINs collected:   1%|▏         | 289/20000 [00:19<30:35, 10.74it/s]



Unique ASINs collected:   2%|▏         | 357/20000 [00:23<16:21, 20.01it/s]

Unique ASINs collected:   2%|▏         | 415/20000 [00:26<13:50, 23.58it/s]

Unique ASINs collected:   2%|▏ 

Done.
Rows: 20054 Unique ASINs: 20054
